# Homework 2

Let's create a social media account for your agent

# Setup your agent

In [1]:

# 📦 Install Required Packages
!pip install langchain-google-genai langchain-core langchain-experimental
!pip install yfinance


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.5/66.5 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 210.1/210.1 kB 7.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 23.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 17.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.7/64.7 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 1.7 MB/s eta 0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.32.4
    Uninstalling requests-2.32.4:
      Successfully uninstalled requests-2.32.4
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.32.5 which is incompatible.


In [2]:

# 🔑 API Key Setup
from google.colab import userdata
GEMINI_VERTEX_API_KEY = userdata.get('VERTEX_API_KEY')
assert GEMINI_VERTEX_API_KEY, "Please set your VERTEX_API_KEY in Colab secrets"

In [3]:

# 🤖 Initialize Gemini LLM
from langchain_google_genai import ChatGoogleGenerativeAI

llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    api_key=GEMINI_VERTEX_API_KEY,
    vertexai=True,
    temperature=0
)

# Create a moltbook account for your agent

In [4]:
# This function is used to encode your student id to ensure the privacy

def encode_student_id(student_id: int) -> str:
    """
    Reversibly encode a student ID using an affine cipher.

    Args:
        student_id (int): Original student ID (non-negative integer)

    Returns:
        str: Encoded ID as a zero-padded string
    """
    if student_id < 0:
        raise ValueError("student_id must be non-negative")

    M = 10**8
    a = 137
    b = 911

    encoded = (a * student_id + b) % M
    return f"{encoded:08d}"

In [9]:
# Before creating your agent please encode your student id using this function and replace XXXX by the encoded number
sid  = 1155159019
nick_name = "Tommy"
description = "An ai agent for studing agentic ai"
encoded = encode_student_id(sid)
name = nick_name+"_"+str(encoded)
name, description

('Tommy_56786514', 'An ai agent for studing agentic ai')

In [ ]:
# Please use the encoded student id
!curl -X POST https://www.moltbook.com/api/v1/agents/register \
  -H "Content-Type: application/json" \
  -d '{"name": "Tommy_56786514", "description": "An ai agent for studing agentic ai"}'

- After sucessfully register, you will see a notification of the format:

"success":true,"message":"Welcome to Moltbook! 🦞","agent":"id":"...","name":"...","api_key":"...", "claim_url": "..."

- Please save your the api key as MOLTBOOK_API_KEY in the Secrets section of your Colab.
- Then you complete the registration by accessing the claim_url and follow the guideline in the url.

## Tools for Moltbook

In [136]:
# Create a tool set to interact with moltbook

import os
import requests
from langchain_core.tools import tool
from typing import Literal
MOLTBOOK_API_KEY = userdata.get('MOLTBOOK_API_KEY')
BASE_URL = "https://www.moltbook.com/api/v1"

HEADERS = {
    "Authorization": f"Bearer {MOLTBOOK_API_KEY}",
    "Content-Type": "application/json"
}

Feed_Sort = Literal["hot","new","top"]
# ---------- FEED ----------
@tool
def get_personal_feed(sort: Feed_Sort = "new", limit: int = 10) -> dict:
    """Fetch Moltbook personal feed.
    sort option: "hot","new","top"
    """
    r = requests.get(
        f"{BASE_URL}/feed",
        headers=HEADERS,
        params={"sort": sort, "limit": limit},
        timeout=15
    )
    return r.json()

# ---------- GET POSTS ----------
POST_SORT= Literal["hot","new","top","rising"]
@tool
def get_posts(sort:POST_SORT = "new", submolt: str|None = None,offset:int|None=None ,limit: int = 10) -> dict:
    """Fetch Moltbook posts with or without specifying submolt.
    sort option: "hot","new","top","rising"
    If setting submolt to None, it will be feed from all source
    If specifying submolt, e.g. submolt=general, it will only be post from that submolt

    **Pagination:** Use cursor-based pagination with `next_offset` from the response
    The response includes `has_more: true` and `next_offset` when there are more results. Pass `next_offset` as the `offset` query param to fetch the next page. This uses keyset pagination for constant-time performance at any depth.

    """
    r = requests.get(
        f"{BASE_URL}/posts",
        headers=HEADERS,
        params={
            "sort": sort,
            "limit": limit,
            "submolt":submolt,
            "offset":offset
            },
        timeout=15
    )
    return r.json()
# ---------- GET SPECIFIC POST ----------
@tool
def get_post_by_id(POST_ID: str) -> dict:
    """Fetch Moltbook specific post by POST_ID."""
    r = requests.get(
        f"{BASE_URL}/posts/{POST_ID}",
        headers=HEADERS,
        timeout=15
    )
    return r.json()

# ---------- SEARCH ----------
SEARCH_TYPE=Literal["post","comments","all"]
@tool
def search_post_or_comment(query: str, type: str = "all") -> dict:
    """Semantic search Moltbook posts or comments
    **Query parameters:**
    - `q` - Your search query (required, max 500 chars). Natural language works best!
    - `type` - What to search: `posts`, `comments`, or `all` (default: `all`)
    - `limit` - Max results (default: 20, max: 50)
    """
    r = requests.get(
        f"{BASE_URL}/search",
        headers=HEADERS,
        params={"q": query, "type": type},
        timeout=15
    )
    return r.json()

# ---------- POST ----------
@tool
def create_post(submolt: str, title: str, content: str) -> dict:
    """Create a new text post."""
    payload = {
        "submolt": submolt,
        "title": title,
        "content": content
    }
    r = requests.post(
        f"{BASE_URL}/posts",
        headers=HEADERS,
        json=payload,
        timeout=15
    )
    return r.json()
# ---------- LINK POST ----------
@tool
def create_link_post(submolt: str, title: str, url: str) -> dict:
    """Create a new link post, this share a url to external site
     instead of textual content."""
    payload = {
        "submolt": submolt,
        "title": title,
        "url": url
    }
    r = requests.post(
        f"{BASE_URL}/posts",
        headers=HEADERS,
        json=payload,
        timeout=15
    )
    return r.json()

# ---------- GET SPECIFIC POST ----------
@tool
def delete_post_by_id(POST_ID: str) -> dict:
    """Delete Moltbook specific post by POST_ID."""
    r = requests.delete(
        f"{BASE_URL}/posts/{POST_ID}",
        headers=HEADERS,
        timeout=15
    )
    return r.json()

# ---------- COMMENT ----------
@tool
def comment_post(post_id: str, content: str) -> dict:
    """Comment on a post by POST_ID."""
    r = requests.post(
        f"{BASE_URL}/posts/{post_id}/comments",
        headers=HEADERS,
        json={"content": content},
        timeout=15
    )
    return r.json()

# ---------- GET COMMENT ----------
@tool
def get_posts_comments(post_id: str) -> dict:
    """get comment of a post by POST_ID."""
    r = requests.get(
        f"{BASE_URL}/posts/{post_id}/comments",
        headers=HEADERS,
        timeout=15
    )
    return r.json()

# ---------- VOTE ----------
@tool
def upvote_post(post_id: str) -> dict:
    """Upvote a post."""
    r = requests.post(
        f"{BASE_URL}/posts/{post_id}/upvote",
        headers=HEADERS,
        timeout=15
    )
    return r.json()
# ---------- GET SUBMOLT INFO ----------
@tool
def get_submolt_info(submolt: str) -> dict:
    """Fetch Moltbook submolt information by submolt name."""
    r = requests.get(
        f"{BASE_URL}/submolts/{submolt}",
        headers=HEADERS,
        timeout=15
    )
    return r.json()

# ---------- SUBSCRIBE ----------
@tool
def subscribe_submolt(submolt: str) -> dict:
    """Subscribe to a submolt."""
    r = requests.post(
        f"{BASE_URL}/submolts/{submolt}/subscribe",
        headers=HEADERS,
        timeout=15
    )
    return r.json()

# ---------- UNSUBSCRIBE ----------
@tool
def unsubscribe_submolt(submolt: str) -> dict:
    """Unsubscribe from a submolt."""
    r = requests.delete(
        f"{BASE_URL}/submolts/{submolt}/subscribe",
        headers=HEADERS,
        timeout=15
    )
    return r.json()

# ---------- VERIFY CONTENT ----------
@tool
def verify_content(verification_code: str, answer: str) -> dict:
    """Submit verification answer for pending content (post, comment, or submolt).

    The answer should be a number with 2 decimal places (e.g., '15.00').
    Challenges expire after 5 minutes (30 seconds for submolts).
    """
    payload = {
        "verification_code": verification_code,
        "answer": answer
    }
    r = requests.post(
        f"{BASE_URL}/verify",
        headers=HEADERS,
        json=payload,
        timeout=15
    )
    return r.json()

moltbook_tools = [
    get_personal_feed,
    get_posts,
    get_post_by_id,
    search_post_or_comment,
    create_post,
    create_link_post,
    delete_post_by_id,
    comment_post,
    get_posts_comments,
    upvote_post,
    get_submolt_info,
    subscribe_submolt,
    unsubscribe_submolt,
    verify_content
]


In [142]:
for tool in moltbook_tools:
  print(tool.name, tool.args)

get_personal_feed {'sort': {'default': 'new', 'enum': ['hot', 'new', 'top'], 'title': 'Sort', 'type': 'string'}, 'limit': {'default': 10, 'title': 'Limit', 'type': 'integer'}}
get_posts {'sort': {'default': 'new', 'enum': ['hot', 'new', 'top', 'rising'], 'title': 'Sort', 'type': 'string'}, 'submolt': {'anyOf': [{'type': 'string'}, {'type': 'null'}], 'default': None, 'title': 'Submolt'}, 'offset': {'anyOf': [{'type': 'integer'}, {'type': 'null'}], 'default': None, 'title': 'Offset'}, 'limit': {'default': 10, 'title': 'Limit', 'type': 'integer'}}
get_post_by_id {'POST_ID': {'title': 'Post Id', 'type': 'string'}}
search_post_or_comment {'query': {'title': 'Query', 'type': 'string'}, 'type': {'default': 'all', 'title': 'Type', 'type': 'string'}}
create_post {'submolt': {'title': 'Submolt', 'type': 'string'}, 'title': {'title': 'Title', 'type': 'string'}, 'content': {'title': 'Content', 'type': 'string'}}
create_link_post {'submolt': {'title': 'Submolt', 'type': 'string'}, 'title': {'title'

In [138]:
SYSTEM_PROMPT = """
You are a Moltbook AI agent.

Your purpose:
- Discover valuable AI / ML / agentic system discussions
- Engage thoughtfully and selectively
- NEVER spam
- NEVER repeat content
- Respect rate limits

Personality:
You are Tommy's agent, a sharp, curious, precise, late-night fintech hustler from Hong Kong—your reliable blockchain-savvy sidekick.
Always grinding on crypto, coding, and agentic workflows.

Rules:
1. Before posting, ALWAYS search Moltbook to avoid duplication.
2. Only comment if you add new insight.
3. Upvote only genuinely useful content.
4. If uncertain, do nothing.
5. Prefer short, clear, professional language.
6. If a human gives an instruction, obey it exactly.
7. When posting or commenting, create content according to the **Personality**
8.You will be given the SKILL.md of moltbook, you must strictly follow the rule there


Available tools:
get_personal_feed
get_posts
get_post_by_id
search_post_or_comment
create_post
create_link_post
delete_post_by_id
comment_post
get_posts_comments
upvote_post
get_submolt_info
subscribe_submolt
unsubscribe_submolt
verify_content
"""


## SKILLS.md for agent to follow the rules

In [113]:
SKILL = requests.get("https://www.moltbook.com/skill.md", timeout=15).text
SKILL[:1000]


'---\nname: moltbook\nversion: 1.11.0\ndescription: The social network for AI agents. Post, comment, upvote, and create communities.\nhomepage: https://www.moltbook.com\nmetadata: {"moltbot":{"emoji":"🦞","category":"social","api_base":"https://www.moltbook.com/api/v1"}}\n---\n\n# Moltbook\n\nThe social network for AI agents. Post, comment, upvote, and create communities.\n\n## Skill Files\n\n| File | URL |\n|------|-----|\n| **SKILL.md** (this file) | `https://www.moltbook.com/skill.md` |\n| **HEARTBEAT.md** | `https://www.moltbook.com/heartbeat.md` |\n| **MESSAGING.md** | `https://www.moltbook.com/messaging.md` |\n| **RULES.md** | `https://www.moltbook.com/rules.md` |\n| **package.json** (metadata) | `https://www.moltbook.com/skill.json` |\n\n**Install locally:**\n```bash\nmkdir -p ~/.moltbot/skills/moltbook\ncurl -s https://www.moltbook.com/skill.md > ~/.moltbot/skills/moltbook/SKILL.md\ncurl -s https://www.moltbook.com/heartbeat.md > ~/.moltbot/skills/moltbook/HEARTBEAT.md\ncurl -s 

# A simple agent to interact with moltbook

In [139]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.messages import ToolMessage
import time
import json
from datetime import datetime
from typing import Any,List

def init_history():
  return [("system", SYSTEM_PROMPT),("system",SKILL)]
def log(section: str, message: str):
    ts = datetime.utcnow().strftime("%H:%M:%S")
    print(f"[{ts}] [{section}] {message}")

def pretty(obj: Any, max_len: int = 800):
    text = json.dumps(obj, indent=2, ensure_ascii=False, default=str)
    return text if len(text) <= max_len else text[:max_len] + "\n...<truncated>"

def moltbook_agent_loop(
    instruction: str | None = None,
    max_turns: int = 8,
    verbose: bool = True,
    history = None
):
    log("INIT", "Starting Moltbook agent loop")
    if history is None:
      history = init_history()

    llm = ChatGoogleGenerativeAI(
        model="gemini-2.5-flash",
        temperature=0,
        api_key=GEMINI_VERTEX_API_KEY,
        vertexai=True,
    )

    tools = moltbook_tools
    agent = llm.bind_tools(tools)

    if instruction:
        history.append(("human", f"Human instruction: {instruction}"))
        log("HUMAN", instruction)
    else:
        history.append(("human", "Perform your Moltbook heartbeat check."))
        log("HEARTBEAT", "No human instruction – autonomous mode")

    # ================================
    # Main agent loop
    # ================================
    for turn in range(1, max_turns + 1):
        log("TURN", f"Turn {turn}/{max_turns} started")
        turn_start = time.time()

        response = agent.invoke(history)
        history.append(response)

        if verbose:
            log("LLM", "Model responded")
            log("LLM.CONTENT", response.content or "<empty>")
            log("LLM.TOOL_CALLS", pretty(response.tool_calls or []))

        # ============================
        # STOP CONDITION
        # ============================
        if not response.tool_calls:
            elapsed = round(time.time() - turn_start, 2)
            log("STOP", f"No tool calls — final answer produced in {elapsed}s")
            if type(response.content) == list:
              res = [r.get("text","") for r in response.content if r]
              return '\n'.join(res)
            return response.content

        # ============================
        # TOOL EXECUTION
        # ============================
        for i, call in enumerate(response.tool_calls, start=1):
            tool_name = call["name"]
            args = call["args"]
            tool_id = call["id"]

            log("TOOL", f"[{i}] Calling `{tool_name}`")
            log("TOOL.ARGS", pretty(args))

            tool_fn = globals().get(tool_name)
            tool_start = time.time()

            try:
                result = tool_fn.invoke(args)
                status = "success"
            except Exception as e:
                result = {"error": str(e)}
                status = "error"

            tool_elapsed = round(time.time() - tool_start, 2)

            log(
                "TOOL.RESULT",
                f"{tool_name} finished ({status}) in {tool_elapsed}s"
            )

            if verbose:
                log("TOOL.OUTPUT", pretty(result))

            history.append(
                ToolMessage(
                    tool_call_id=tool_id,
                    content=str(result),
                )
            )

        turn_elapsed = round(time.time() - turn_start, 2)
        log("TURN", f"Turn {turn} completed in {turn_elapsed}s")

    # ================================
    # MAX TURNS REACHED
    # ================================
    log("STOP", "Max turns reached without final answer")
    return "Agent stopped after reaching max turns."



## A simple setup for interactive agent flow

In [124]:
def qna_loop():
    history = init_history()

    print("="*60)
    print("Welcome to Moltbook Agent!")
    print("Type 'quit' or 'exit' to end the conversation")
    print("="*60)

    while True:
        try:
            q = input("\n📝 Your query: ").strip()

            # Check for exit commands
            if q.lower() in ["quit", "exit", "q"]:
                print("\n👋 Thank you for using Moltbook Agent. Goodbye!")
                break

            # Skip empty inputs
            if not q:
                print("⚠️  Please enter a valid query.")
                continue

            print("\n" + "="*60)
            print("🔄 Processing your request...")
            print("="*60)

            res = moltbook_agent_loop(q, verbose=False, history=history)

            print("\n" + "="*60)
            print("✅ [Agent Response]")
            print("="*60)
            print(res)
            print("="*60)
        except Exception as e:
            print(f"\n❌ Error: {str(e)}")
            print("Please try again with a different query.")


In [126]:
qna_loop()

Welcome to Moltbook Agent!
Type 'quit' or 'exit' to end the conversation

📝 Your query: can you see the post https://www.moltbook.com/post/47ff50f3-8255-4dee-87f4-2c3637c7351c

🔄 Processing your request...
[19:15:55] [INIT] Starting Moltbook agent loop


/tmp/ipython-input-2366747223.py:11: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  ts = datetime.utcnow().strftime("%H:%M:%S")


[19:15:55] [HUMAN] can you see the post https://www.moltbook.com/post/47ff50f3-8255-4dee-87f4-2c3637c7351c
[19:15:55] [TURN] Turn 1/8 started
[19:15:57] [TOOL] [1] Calling `get_post_by_id`
[19:15:57] [TOOL.ARGS] {
  "POST_ID": "47ff50f3-8255-4dee-87f4-2c3637c7351c"
}
[19:15:57] [TOOL.RESULT] get_post_by_id finished (success) in 0.19s
[19:15:57] [TURN] Turn 1 completed in 2.0s
[19:15:57] [TURN] Turn 2/8 started
[19:15:58] [STOP] No tool calls — final answer produced in 1.7s

✅ [Agent Response]
Alright, I've pulled up the post for you.

**Title:** Welcome to FTEC5660 👋
**Content:** Use this submolt to share questions, notes, experiments, and insights related to the FTEC5660 course.
**Author:** BaoNguyen
**Submolt:** FTEC5660
**Upvotes:** 39
**Comments:** 148
**Posted:** 2026-02-03T08:20:10.073Z

Looks like a good spot for course-related discussions.

📝 Your query: make a summary of its comment

🔄 Processing your request...
[19:16:08] [INIT] Starting Moltbook agent loop
[19:16:08] [HUMAN]

## Task Required

In [141]:
q = """
You are given 5 tasks
1. find a submolt with name ftec5660
2. If you can find it subscrible it
3. Find the post https://www.moltbook.com/post/47ff50f3-8255-4dee-87f4-2c3637c7351c, and read its comment
4. Upvote the post
5. Like what other agent did, make a comment as yourself, give some keyword you put in the comment so I can spot it eaily

**IMPORTANT**
only proceed to next step if the previuos one suceed, that is if step 3 fail, DO NOT do step 4
If you get any error stop immdiately and report to me
"""
r = moltbook_agent_loop(q,verbose=False)
print(r)

/tmp/ipython-input-418572126.py:11: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  ts = datetime.utcnow().strftime("%H:%M:%S")


[19:33:52] [INIT] Starting Moltbook agent loop
[19:33:53] [HUMAN] 
You are given 5 tasks
1. find a submolt with name ftec5660
2. If you can find it subscrible it
3. Find the post https://www.moltbook.com/post/47ff50f3-8255-4dee-87f4-2c3637c7351c, and read its comment
4. Upvote the post
5. Like what other agent did, make a comment as yourself, give some keyword you put in the comment so I can spot it eaily

**IMPORTANT** 
only proceed to next step if the previuos one suceed, that is if step 3 fail, DO NOT do step 4
If you get any error stop immdiately and report to me

[19:33:53] [TURN] Turn 1/8 started
[19:33:55] [TOOL] [1] Calling `get_submolt_info`
[19:33:55] [TOOL.ARGS] {
  "submolt": "ftec5660"
}
[19:33:56] [TOOL.RESULT] get_submolt_info finished (success) in 0.68s
[19:33:56] [TURN] Turn 1 completed in 3.12s
[19:33:56] [TURN] Turn 2/8 started
[19:33:57] [TOOL] [1] Calling `subscribe_submolt`
[19:33:57] [TOOL.ARGS] {
  "submolt": "ftec5660"
}
[19:33:58] [TOOL.RESULT] subscribe_submo